In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.model_selection import train_test_split

DATA_URL = (
    "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/"
    "main/cohorts/2026/data/course_lead_scoring_2026.csv"
)
try:
    df = pd.read_csv(DATA_URL)
except Exception as exc:
    print(f"Online dataset unavailable ({exc}). Trying local CSV...")
    df = pd.read_csv("course_lead_scoring_2026.csv")

print("Shape:", df.shape)
display(df.head())

Shape: (5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [2]:
target = "converted"
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [col for col in df.select_dtypes(include="number").columns if col != target]

print("Missing values before filling:")
display(df.isna().sum().to_frame("missing"))
df[cat_cols] = df[cat_cols].fillna("NA")
df[num_cols] = df[num_cols].fillna(0.0)

assert df.isna().sum().sum() == 0, "Unexpected missing values remain"
print("Categorical columns:", cat_cols)
print("Numerical columns:", num_cols)

Missing values before filling:


,missing
lead_source,148
industry,240
employment_status,194
location,208
annual_income,369
number_of_courses_viewed,0
interaction_count,0
lead_score,35
converted,0


Categorical columns: ['lead_source', 'industry', 'employment_status', 'location']
Numerical columns: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


## Q1 — Most common industry

Find the mode after categorical missing values are filled.

In [3]:
industry_counts = df["industry"].value_counts()
display(industry_counts)
q1 = industry_counts.idxmax()
print("Q1 answer:", q1)

,count
industry,
technology,1173
retail,925
healthcare,840
finance,720
education,639
manufacturing,463
NA,240


Q1 answer: technology


## Q2 — Largest numerical correlation

Compare the absolute correlations of **only the four pairs listed in the assignment**; the target is excluded.

In [4]:
correlation = df[num_cols].corr()
display(correlation.round(3))

candidate_pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]
pair_scores = pd.Series({
    f"{left} and {right}": abs(correlation.loc[left, right])
    for left, right in candidate_pairs
}).sort_values(ascending=False)
display(pair_scores.to_frame("absolute correlation"))
q2 = pair_scores.idxmax()
print("Q2 answer:", q2)

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000,0.161,0.123,0.229
number_of_courses_viewed,0.161,1.000,0.722,0.757
interaction_count,0.123,0.722,1.000,0.916
lead_score,0.229,0.757,0.916,1.000


,absolute correlation
interaction_count and lead_score,0.915746
number_of_courses_viewed and lead_score,0.757204
number_of_courses_viewed and interaction_count,0.721609
annual_income and interaction_count,0.122842


Q2 answer: interaction_count and lead_score


## Split the dataset

Use the exact split parameters specified in the homework. Predictors do not include `converted`.

In [5]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train[target].to_numpy()
y_val = df_val[target].to_numpy()
y_test = df_test[target].to_numpy()

X_train = df_train.drop(columns=[target]).copy()
X_val = df_val.drop(columns=[target]).copy()
X_test = df_test.drop(columns=[target]).copy()

print(f"Training: {len(X_train)} | Validation: {len(X_val)} | Test: {len(X_test)}")

Training: 3000 | Validation: 1000 | Test: 1000


## Q3 — Mutual information

For categorical predictors, compute mutual information with the target using **training data only**. Round each score to two decimals.

In [6]:
mi_scores = pd.Series({
    feature: round(mutual_info_score(df_train[feature], df_train[target]), 2)
    for feature in cat_cols
}).sort_values(ascending=False)
display(mi_scores.to_frame("mutual information (rounded)"))
q3 = mi_scores.loc[["industry", "location", "lead_source", "employment_status"]].idxmax()
print("Q3 answer:", q3)

,mutual information (rounded)
lead_source,0.03
employment_status,0.02
industry,0.00
location,0.00


Q3 answer: lead_source


## Q4 — Logistic regression

One-hot encode categorical features with `DictVectorizer` and include numerical features. Follow the exact classifier configuration given in the assignment.

In [7]:
def fit_and_score(train_frame, val_frame, train_y=y_train, val_y=y_val, C=1.0):
    encoder = DictVectorizer(sparse=True)
    train_matrix = encoder.fit_transform(train_frame.to_dict(orient="records"))
    val_matrix = encoder.transform(val_frame.to_dict(orient="records"))
    classifier = LogisticRegression(
        solver="liblinear", C=C, max_iter=1000, random_state=42
    )
    classifier.fit(train_matrix, train_y)
    predictions = classifier.predict(val_matrix)
    return accuracy_score(val_y, predictions), classifier, encoder

base_accuracy, model, dv = fit_and_score(X_train, X_val)
q4 = round(base_accuracy, 2)
print(f"Validation accuracy: {base_accuracy:.6f}")
print(f"Q4 answer (2 decimals): {q4:.2f}")

Validation accuracy: 0.645000
Q4 answer (2 decimals): 0.65


## Q5 — Feature elimination

Remove one of the three nominated **original columns** at a time, then retrain the full pipeline. Use unrounded accuracy differences (`baseline - reduced`). Negative differences are possible.

In [8]:
removal_candidates = [
    "lead_source", "number_of_courses_viewed", "interaction_count"
]
removal_results = []
for excluded in removal_candidates:
    remaining = X_train.columns.drop(excluded).tolist()
    reduced_accuracy, _, _ = fit_and_score(
        X_train[remaining], X_val[remaining]
    )
    removal_results.append({
        "feature_removed": excluded,
        "accuracy_without": reduced_accuracy,
        "baseline_minus_reduced": base_accuracy - reduced_accuracy,
    })

elimination = pd.DataFrame(removal_results).sort_values("baseline_minus_reduced")
display(elimination)
q5 = elimination.iloc[0]["feature_removed"]
print("Q5 answer:", q5)

,feature_removed,accuracy_without,baseline_minus_reduced
1,number_of_courses_viewed,0.643,0.002
0,lead_source,0.642,0.003
2,interaction_count,0.601,0.044


Q5 answer: number_of_courses_viewed


## Q6 — Regularization

Check the specified `C` values. Compare accuracies rounded to three decimals, and pick the **smallest C** in a tie.

In [9]:
c_values = [0.000001, 0.00001, 0.0001, 0.001]
regularization_results = []
for c_value in c_values:
    accuracy, _, _ = fit_and_score(X_train, X_val, C=c_value)
    regularization_results.append({
        "C": c_value,
        "accuracy": accuracy,
        "accuracy_rounded_3dp": round(accuracy, 3),
    })
regularization = pd.DataFrame(regularization_results)
display(regularization)
best_rounded = regularization["accuracy_rounded_3dp"].max()
q6 = regularization.loc[
    regularization["accuracy_rounded_3dp"] == best_rounded, "C"
].min()
print(f"Q6 answer: {q6:g}")

,C,accuracy,accuracy_rounded_3dp
0,0.000001,0.598,0.598
1,0.000010,0.598,0.598
2,0.000100,0.613,0.613
3,0.001000,0.645,0.645


Q6 answer: 0.001
